# executor

> Approval-gated python execution for the harness. The default is a **persistent Jupyter kernel**
> (via `conkernelclient`): a separate interruptible process, not an in-process `exec` with guards.

In [ ]:
#| default_exp executor

In [ ]:
#| hide
from nbdev.showdoc import *

Three layers, deliberately separate:

- **approval** happens host-side before anything runs: every snippet is offered to `approve` in the
  same `{'function': {'name': ..., 'arguments': ...}}` shape rishi's tool loop uses, so
  `rishi.core.hitl_policy({'run_py': 'check'})` works unchanged.
- **execution** happens in a real kernel process (`KernelExecutor`, the default). That buys process
  isolation, `interrupt()` for runaway cells, `restart()` for a genuinely fresh interpreter, and
  timeouts -- none of which in-process `exec` can offer. It's a stock ipykernel by default, or pass
  `connection_file=` to attach to an already-running kernel (e.g. a conkernel session server).
- **fine-grained policy**, when you want it, belongs *inside* the kernel: `safepyrun` is built
  directly on pyskills' `allow()`/`__pytools__` registry, so skills that declare `allow()` grants
  compose with it -- load it in the kernel rather than reinventing guards out here.

rishi's own `run_py` runs through `safepyrun`'s sandbox, which blocks `socket` and `importlib` --
correct for untrusted one-shots, fatal for skills whose whole point is the network (fossick) or
imports (loading pyskills). Hence this module.

The kernel's namespace persists across calls -- and across conversation compaction, since it lives
in the kernel process and not in the model's context.

`Executor` (in-process `exec`, same interface and gate) remains as a fallback for tests and
environments without a kernel; `default_executor()` picks the kernel when it can and falls back
otherwise.

In [ ]:
#| export
import io, ast, asyncio, threading
from contextlib import redirect_stdout

In [ ]:
#| export
def code_call(code):
    'Wrap a snippet in the tool-call dict shape rishi\'s `approve`/`hitl_policy` expect.'
    return {'function': {'name': 'run_py', 'arguments': {'code': code}}}

In [ ]:
#| export
class Executor:
    'Run python snippets in one persistent namespace, gated by `approve`, returning stdout + last-expr repr.'
    def __init__(self,
                 ns:dict=None,      # namespace to run in (created if None); persists across calls
                 approve=None,      # approve(tool_call)->bool, rishi-compatible; None = allow all
                 max_len:int=4000): # truncate returned output beyond this many chars
        self.ns = ns if ns is not None else {}
        self.ns.setdefault('__name__', '__ramabana__')
        self.approve,self.max_len = approve,max_len
    def __call__(self, code:str) -> str:
        if self.approve is not None and not self.approve(code_call(code)): return 'Denied by human operator'
        buf = io.StringIO()
        try:
            tree = ast.parse(code)
            last = tree.body[-1] if tree.body and isinstance(tree.body[-1], ast.Expr) else None
            body = tree.body[:-1] if last is not None else tree.body
            with redirect_stdout(buf):
                if body: exec(compile(ast.Module(body=body, type_ignores=[]), '<ramabana>', 'exec'), self.ns)
                res = eval(compile(ast.Expression(body=last.value), '<ramabana>', 'eval'), self.ns) if last is not None else None
            out = buf.getvalue()
            if res is not None: out = (out.rstrip('\n') + '\n' if out else '') + repr(res)
            out = out or '(ok)'
        except Exception as e:
            pre = buf.getvalue()
            out = (pre + '\n' if pre else '') + f'{type(e).__name__}: {e}'
        if len(out) > self.max_len: out = out[:self.max_len] + f'\n... [truncated {len(out)-self.max_len} chars]'
        return out

In [ ]:
ex = Executor()
assert ex('x = 2') == '(ok)'
assert ex('x + 1') == '3'                          # namespace persists, last expr repr'd
assert ex('print("hi"); x*2') == 'hi\n4'           # stdout + repr
assert ex('1/0').startswith('ZeroDivisionError')   # exceptions come back as text, loop continues
assert ex('import json; json.dumps({"a": 1})') == '\'{"a": 1}\''

In [ ]:
# approval gate: rishi-style policy dicts work as-is
deny = Executor(approve=lambda tc: tc['function']['name'] != 'run_py')
assert deny('x = 1') == 'Denied by human operator' and 'x' not in deny.ns
seen = []
audit = Executor(approve=lambda tc: seen.append(tc['function']['arguments']['code']) or True)
assert audit('7*6') == '42' and seen == ['7*6']

In [ ]:
long = Executor(max_len=10)
assert long('"a"*50').startswith("'aaaaaaaaa") and 'truncated' in long('"a"*50')

## KernelExecutor

The default executor: one long-lived Jupyter kernel per session, driven through
[`conkernelclient`](https://github.com/AnswerDotAI/conkernelclient)'s concurrent-safe client. The
client's asyncio machinery runs on a dedicated background thread, so the executor presents the same
plain synchronous `__call__(code) -> str` the harness expects.

In [ ]:
#| export
def render_outputs(outs):
    'nbformat-style outputs (from `exec_outs`) as plain text for a ```result fence.'
    parts = []
    for o in outs:
        t = o['output_type']
        if t == 'stream': parts.append(o.get('text',''))
        elif t in ('execute_result','display_data'):
            txt = o.get('data',{}).get('text/plain')
            if txt is not None: parts.append(txt if isinstance(txt,str) else ''.join(txt))
        elif t == 'error': parts.append(f"{o.get('ename','Error')}: {o.get('evalue','')}")
    return '\n'.join(p.rstrip('\n') for p in parts if p and p.strip()) or '(ok)'

In [ ]:
#| export
class KernelExecutor:
    'Run python in a persistent Jupyter kernel process, gated by `approve`; interruptible and restartable.'
    def __init__(self,
                 approve=None,           # approve(tool_call)->bool, rishi-compatible; None = allow all
                 kernel_name='python3',  # kernelspec to launch (ignored with connection_file)
                 connection_file=None,   # attach to an already-running kernel (e.g. a conkernel server) instead of launching
                 timeout:int=60,         # per-execution seconds before the cell is interrupted
                 max_len:int=4000,       # truncate returned output beyond this many chars
                 cwd=None):              # working directory for a launched kernel
        from conkernelclient.core import ConKernelClient, ConKernelManager
        self.approve,self.timeout,self.max_len,self.km = approve,timeout,max_len,None
        self._loop = asyncio.new_event_loop()
        self._thread = threading.Thread(target=self._loop.run_forever, daemon=True)
        self._thread.start()
        if connection_file:
            async def _attach():
                kc = ConKernelClient()
                kc.load_connection_file(connection_file)
                return await kc.start_channels()
            self.kc = self._sync(_attach(), timeout=30)
        else:
            async def _start():
                km = ConKernelManager(kernel_name=kernel_name)
                await km.start_kernel(**({'cwd': str(cwd)} if cwd else {}))
                return km, await km.client().start_channels()
            self.km, self.kc = self._sync(_start(), timeout=60)
    def _sync(self, coro, timeout=None):
        'Run `coro` on the client\'s event-loop thread and block for its result.'
        return asyncio.run_coroutine_threadsafe(coro, self._loop).result(timeout)
    def __call__(self, code:str) -> str:
        if self.approve is not None and not self.approve(code_call(code)): return 'Denied by human operator'
        try: outs = self._sync(self.kc.exec_outs(code, timeout=self.timeout), timeout=self.timeout+10)
        except TimeoutError:
            try:
                self._sync(self.kc.interrupt(), timeout=10)
                self._sync(self.kc.iopub_flush(0.5), timeout=10)
            except Exception: pass
            return f'[interrupted: no result within {self.timeout}s]'
        out = render_outputs(outs)
        if len(out) > self.max_len: out = out[:self.max_len] + f'\n... [truncated {len(out)-self.max_len} chars]'
        return out
    def eval(self, expr:str):
        'Evaluate `expr` in the kernel and return its (literal_eval-parsed) value; bypasses the gate -- host use only.'
        return self._sync(self.kc.eval_expr(expr, timeout=self.timeout), timeout=self.timeout+10)
    def interrupt(self):
        'Interrupt whatever the kernel is running, keeping session state.'
        return self._sync(self.kc.interrupt(), timeout=10)
    def restart(self):
        'Fresh kernel process (new pid, state discarded); only for kernels this executor launched.'
        if self.km is None: raise RuntimeError('cannot restart an attached kernel')
        from conkernelclient.ops import reconnect
        self.kc = self._sync(reconnect(self.km, self.kc), timeout=60)
    def close(self):
        'Stop channels, shut down a launched kernel, and stop the background loop.'
        async def _stop():
            self.kc.stop_channels()
            if self.km is not None: await self.km.shutdown_kernel(now=True)
        try: self._sync(_stop(), timeout=15)
        finally: self._loop.call_soon_threadsafe(self._loop.stop)

In [ ]:
#| export
def default_executor(approve=None, **kw):
    'A `KernelExecutor` when a kernel can be started, else the in-process `Executor` fallback.'
    try: return KernelExecutor(approve=approve, **kw)
    except Exception: return Executor(approve=approve)

In [ ]:
ke = KernelExecutor(timeout=5)
assert ke('x = 2') == '(ok)'
assert ke('x + 1') == '3'
assert ke('print("hi"); x*2') == 'hi\n4'
assert ke('1/0').startswith('ZeroDivisionError')

In [ ]:
# the gate runs host-side: denied code never reaches the kernel
ke.approve = lambda tc: tc['function']['name'] != 'run_py'
assert ke('x = 99') == 'Denied by human operator'
ke.approve = None
assert ke.eval('x') == 2

In [ ]:
# a runaway cell is interrupted at `timeout`, and the kernel stays usable
assert ke('import time; time.sleep(30); "done"').startswith('[interrupted')
assert ke('x + 40') == '42'

In [ ]:
# restart() is a genuinely fresh interpreter
ke.restart()
assert ke('"x" in dir()') == 'False'
ke.close()

In [ ]:
de = default_executor()
assert isinstance(de, (KernelExecutor, Executor))
if isinstance(de, KernelExecutor): assert de('21*2') == '42'; de.close()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()